# 🎫 Support Tickets App - Lakebase Setup

This notebook sets up the Lakebase infrastructure required for the Support Tickets Dash application.

**What this notebook does:**
1. Creates a Lakebase database instance
2. Registers a Unity Catalog for the database
3. Connects to PostgreSQL and creates the schema/table
4. Seeds 2-3 sample tickets for each user in the workspace

Run this notebook before deploying the Support Tickets app.

---
## Step 1: Install Dependencies

Install the required packages and restart the Python environment.

In [ ]:
%pip install --upgrade databricks-sdk psycopg2-binary -q
dbutils.library.restartPython()

---
## Step 2: Configuration

Configure the Lakebase instance and table settings.

In [ ]:
# Lakebase configuration
LAKEBASE_INSTANCE_NAME = "support-tickets-lakebase"
LAKEBASE_CATALOG_NAME = "support_tickets_catalog"

# PostgreSQL schema and table
PG_SCHEMA = "public"
PG_TABLE = "support_tickets"

# Number of sample tickets to create per user
TICKETS_PER_USER = 3

print(f"✅ Configuration loaded")
print(f"   Instance: {LAKEBASE_INSTANCE_NAME}")
print(f"   Catalog:  {LAKEBASE_CATALOG_NAME}")
print(f"   Schema:   {PG_SCHEMA}")
print(f"   Table:    {PG_TABLE}")
print(f"   Tickets per user: {TICKETS_PER_USER}")

---
## Step 3: Create Lakebase Instance

A Lakebase instance is a Databricks-managed PostgreSQL database.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import (
    DatabaseInstance, 
    DatabaseCatalog, 
    DatabaseInstanceState
)

# Initialize the Databricks workspace client
w = WorkspaceClient()

# Configure the database instance
db_instance_config = DatabaseInstance(
    name=LAKEBASE_INSTANCE_NAME,
    capacity="CU_2",
)

# Create a new Lakebase database instance
try:
    instance = w.database.create_database_instance(db_instance_config)
    print(f"✅ Instance created: {db_instance_config.name}")
    print(f"   Please wait a few minutes for the instance to become available.")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"ℹ️  Instance '{db_instance_config.name}' already exists")
    else:
        print(f"❌ Error creating instance: {e}")

---
## Step 4: Wait for Instance to be Available

The Lakebase instance takes a few minutes to provision.

In [ ]:
import time

def wait_for_instance(instance_name, timeout_minutes=10):
    """Wait for the Lakebase instance to become available."""
    print(f"Waiting for instance '{instance_name}' to be available...")
    
    start_time = time.time()
    timeout_seconds = timeout_minutes * 60
    
    while True:
        try:
            instance = w.database.get_database_instance(instance_name)
            state = instance.state
            
            if state == DatabaseInstanceState.AVAILABLE:
                print(f"\n✅ Instance is AVAILABLE!")
                return True
            
            elapsed = int(time.time() - start_time)
            print(f"   Status: {state} (elapsed: {elapsed}s)", end="\r")
            
            if elapsed > timeout_seconds:
                print(f"\n⏰ Timeout after {timeout_minutes} minutes. Current state: {state}")
                return False
            
            time.sleep(10)
            
        except Exception as e:
            print(f"\n❌ Error checking instance: {e}")
            return False

# Wait for instance
instance_ready = wait_for_instance(LAKEBASE_INSTANCE_NAME)

---
## Step 5: Register Database Catalog

Register a Unity Catalog for the Lakebase instance.

In [ ]:
# Check if instance is available first
instance = w.database.get_database_instance(LAKEBASE_INSTANCE_NAME)

if instance.state != DatabaseInstanceState.AVAILABLE:
    print(f"⚠️  Instance is not ready yet. Current state: {instance.state}")
    print(f"   Please wait and run this cell again.")
else:
    # Configure the catalog
    catalog_config = DatabaseCatalog(
        name=LAKEBASE_CATALOG_NAME,
        database_instance_name=LAKEBASE_INSTANCE_NAME,
        database_name="databricks_postgres",
    )
    
    # Register the catalog
    try:
        catalog = w.database.create_database_catalog(catalog_config)
        print(f"✅ Created database catalog: {catalog_config.name}")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"ℹ️  Catalog '{catalog_config.name}' already exists")
        else:
            print(f"❌ Error creating catalog: {e}")

---
## Step 6: Connect to PostgreSQL

Connect directly to the Lakebase PostgreSQL instance using generated credentials.

In [ ]:
import psycopg2
import uuid

# Get instance details
instance = w.database.get_database_instance(name=LAKEBASE_INSTANCE_NAME)

# Generate database credential
cred = w.database.generate_database_credential(
    request_id=str(uuid.uuid4()), 
    instance_names=[LAKEBASE_INSTANCE_NAME]
)

# Connection parameters
pg_host = instance.read_write_dns
pg_database = "databricks_postgres"
pg_user = w.current_user.me().user_name
pg_password = cred.token

print(f"📋 PostgreSQL Connection Info:")
print(f"   Host: {pg_host}")
print(f"   Database: {pg_database}")
print(f"   User: {pg_user}")

In [ ]:
# Create connection to PostgreSQL
conn = psycopg2.connect(
    host=pg_host,
    dbname=pg_database,
    user=pg_user,
    password=pg_password,
    sslmode="require"
)
conn.autocommit = True
cursor = conn.cursor()

# Verify connection
cursor.execute("SELECT version()")
version = cursor.fetchone()[0]
print(f"✅ Connected to PostgreSQL!")
print(f"   {version}")

---
## Step 7: Create Table

Create the support_tickets table in PostgreSQL.

In [ ]:
# Create table
create_table_sql = f"""
CREATE TABLE IF NOT EXISTS {PG_SCHEMA}.{PG_TABLE} (
    id BIGSERIAL PRIMARY KEY,
    title VARCHAR(255) NOT NULL,
    description TEXT NOT NULL,
    customer_email VARCHAR(255) NOT NULL,
    status VARCHAR(50) NOT NULL DEFAULT 'open',
    priority VARCHAR(20) NOT NULL DEFAULT 'medium',
    assigned_to VARCHAR(255) NOT NULL,
    created_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    updated_at TIMESTAMPTZ NOT NULL DEFAULT now()
);
"""

try:
    cursor.execute(create_table_sql)
    print(f"✅ Created table: {PG_SCHEMA}.{PG_TABLE}")
except Exception as e:
    print(f"❌ Error creating table: {e}")

---
## Step 8: Get Workspace Users

Fetch all users in the workspace to seed tickets for each one.

In [ ]:
# Get all users in the workspace
users = list(w.users.list())

# Filter to active users with email addresses
active_users = [
    u for u in users 
    if u.active and u.user_name and '@' in u.user_name
]

print(f"📋 Found {len(active_users)} active users in workspace:")
for user in active_users[:10]:  # Show first 10
    print(f"   - {user.user_name}")
if len(active_users) > 10:
    print(f"   ... and {len(active_users) - 10} more")

---
## Step 9: Seed Sample Tickets

Create 2-3 sample tickets for each user in the workspace.

In [ ]:
import random

# Sample ticket templates
ticket_templates = [
    {
        "title": "Cannot login to dashboard",
        "description": "I keep getting an 'Invalid credentials' error when trying to log in. I've reset my password twice but still can't access my account.",
        "priority": "high",
        "status": "open"
    },
    {
        "title": "Report export is slow",
        "description": "Exporting quarterly reports takes over 10 minutes. This used to complete in under a minute. Please investigate.",
        "priority": "medium",
        "status": "in_progress"
    },
    {
        "title": "Data sync not working",
        "description": "The data sync between our CRM and your platform stopped working yesterday. We're missing critical customer updates.",
        "priority": "critical",
        "status": "open"
    },
    {
        "title": "Feature request: Dark mode",
        "description": "Would love to have a dark mode option in the dashboard. It would be easier on the eyes during late night work sessions.",
        "priority": "low",
        "status": "open"
    },
    {
        "title": "Billing discrepancy",
        "description": "I was charged twice for my monthly subscription. Please review and issue a refund for the duplicate charge.",
        "priority": "high",
        "status": "resolved"
    },
    {
        "title": "API rate limiting issues",
        "description": "We're hitting rate limits much sooner than expected based on our plan. Can you check if there's an issue with our quota?",
        "priority": "medium",
        "status": "in_progress"
    },
    {
        "title": "Mobile app crashes on startup",
        "description": "After the latest update, the iOS app crashes immediately upon opening. iPhone 14 Pro, iOS 17.2.",
        "priority": "critical",
        "status": "open"
    },
    {
        "title": "Need help with integration",
        "description": "We're trying to integrate your API with our Salesforce instance. Documentation seems outdated. Can someone help?",
        "priority": "medium",
        "status": "closed"
    },
    {
        "title": "Password reset email not received",
        "description": "Requested password reset 3 times, checked spam folder, still no email. Need access urgently for a client meeting.",
        "priority": "high",
        "status": "resolved"
    },
    {
        "title": "Dashboard widgets not loading",
        "description": "The analytics widgets on my dashboard show 'Loading...' indefinitely. Tried clearing cache and different browsers.",
        "priority": "medium",
        "status": "in_progress"
    }
]

# Customer email domains for variety
customer_domains = ["acme.com", "techcorp.io", "enterprise.co", "startup.io", "business.net"]

print(f"📝 Ticket templates ready: {len(ticket_templates)} templates")

In [ ]:
# Seed tickets for each user
total_inserted = 0

for user in active_users:
    user_email = user.user_name
    
    # Select random tickets for this user
    user_tickets = random.sample(ticket_templates, min(TICKETS_PER_USER, len(ticket_templates)))
    
    for ticket in user_tickets:
        # Generate a random customer email
        customer_name = f"customer{random.randint(100, 999)}"
        customer_domain = random.choice(customer_domains)
        customer_email = f"{customer_name}@{customer_domain}"
        
        # Insert ticket
        insert_sql = f"""
            INSERT INTO {PG_SCHEMA}.{PG_TABLE} 
            (title, description, customer_email, status, priority, assigned_to)
            VALUES (%s, %s, %s, %s, %s, %s)
        """
        cursor.execute(insert_sql, (
            ticket["title"],
            ticket["description"],
            customer_email,
            ticket["status"],
            ticket["priority"],
            user_email
        ))
        total_inserted += 1

print(f"✅ Inserted {total_inserted} tickets for {len(active_users)} users")
print(f"   ({TICKETS_PER_USER} tickets per user)")

---
## Step 10: Verify Data

Query the table to verify the data was inserted correctly.

In [ ]:
# Check total tickets
cursor.execute(f"SELECT COUNT(*) FROM {PG_SCHEMA}.{PG_TABLE}")
total_count = cursor.fetchone()[0]
print(f"📊 Total tickets in database: {total_count}")

# Show distribution by user (first 10)
cursor.execute(f"""
    SELECT assigned_to, COUNT(*) as ticket_count 
    FROM {PG_SCHEMA}.{PG_TABLE} 
    GROUP BY assigned_to 
    ORDER BY ticket_count DESC
    LIMIT 10
""")
print(f"\n📋 Tickets by user (top 10):")
for row in cursor.fetchall():
    print(f"   {row[0]}: {row[1]} tickets")

In [ ]:
# Show sample tickets
cursor.execute(f"""
    SELECT id, title, status, priority, assigned_to 
    FROM {PG_SCHEMA}.{PG_TABLE} 
    ORDER BY id 
    LIMIT 10
""")
print(f"📋 Sample tickets:")
print(f"{'ID':<5} {'Title':<35} {'Status':<12} {'Priority':<10} {'Assigned To'}")
print("-" * 100)
for row in cursor.fetchall():
    print(f"{row[0]:<5} {row[1][:33]:<35} {row[2]:<12} {row[3]:<10} {row[4]}")

In [ ]:
# Close connection
cursor.close()
conn.close()
print("✅ PostgreSQL connection closed")

---
## ✅ Setup Complete!

Your Lakebase instance is ready with:
- **Instance:** `support-tickets-lakebase`
- **Catalog:** `support_tickets_catalog`
- **Schema:** `public`
- **Table:** `support_tickets`
- Sample tickets seeded for each workspace user

### Deploy the App

Update your `app.yml` with:

```yaml
command:
  - python
  - app.py

resources:
  - name: support-tickets-db
    database:
      instance: support-tickets-lakebase
      catalog: support_tickets_catalog
```

Then deploy:
```bash
databricks apps deploy
```

In [ ]:
print("🎉 Lakebase setup complete!")
print("")
print(f"Summary:")
print(f"  🗄️ Instance: {LAKEBASE_INSTANCE_NAME}")
print(f"  📚 Catalog: {LAKEBASE_CATALOG_NAME}")
print(f"  📋 Table: {PG_SCHEMA}.{PG_TABLE}")
print(f"  📊 Total tickets: {total_inserted}")
print(f"  👥 Users with tickets: {len(active_users)}")
print("")
print("Next steps:")
print("1. Update app.yml with the database resource configuration")
print("2. Deploy the app: databricks apps deploy")

---
## 🧹 Cleanup (Optional)

Run these cells to clean up resources if needed. **Warning:** This will delete all data!

In [ ]:
# ⚠️ DANGER: Uncomment and run to delete all resources

# # Reconnect to drop the table
# cred = w.database.generate_database_credential(
#     request_id=str(uuid.uuid4()), 
#     instance_names=[LAKEBASE_INSTANCE_NAME]
# )
# conn = psycopg2.connect(
#     host=pg_host,
#     dbname=pg_database,
#     user=pg_user,
#     password=cred.token,
#     sslmode="require"
# )
# conn.autocommit = True
# cursor = conn.cursor()

# # Drop table
# cursor.execute(f"DROP TABLE IF EXISTS {PG_SCHEMA}.{PG_TABLE}")
# print(f"🗑️ Dropped table: {PG_SCHEMA}.{PG_TABLE}")

# cursor.close()
# conn.close()

# # Delete catalog
# try:
#     w.database.delete_database_catalog(LAKEBASE_CATALOG_NAME)
#     print(f"🗑️ Deleted catalog: {LAKEBASE_CATALOG_NAME}")
# except Exception as e:
#     print(f"Error deleting catalog: {e}")

# # Delete instance
# try:
#     w.database.delete_database_instance(LAKEBASE_INSTANCE_NAME)
#     print(f"🗑️ Deleted instance: {LAKEBASE_INSTANCE_NAME}")
# except Exception as e:
#     print(f"Error deleting instance: {e}")